# The resolution gap in many degrees of freedom: check it yourself

An observer sees a Hamiltonian ensemble through a Gaussian blur of width $\sigma$. The coarse-grained entropy can only grow if the blur widens faster than a critical rate $R$ of the state, and $R$ never exceeds the largest stretching rate $\kappa$ of the flow. At finite resolution the best Gaussian state stays below $\kappa$ by a margin, the **resolution gap**.

The paper [1] gives a single matrix formula for this gap in any number of degrees of freedom, for $H = |p|^2/2 + V(q)$:
$$\kappa - R^* \;\approx\; \Gamma\,\sigma, \qquad \Gamma = 2\,\mathrm{tr}\Big[\big(B^{1/2} C B^{1/2}\big)^{1/2}\Big],$$
where $B$ says how fast the maximal strain falls when the state spreads in configuration space, and $C = \mathrm{diag}(2\kappa, \kappa, \dots, \kappa)$ is the price of spreading the observer's focus.

This notebook checks four claims of the paper on **two coupled pendulums** and pendulum chains, from scratch, in about two minutes on a CPU:

1. $\Gamma$ computed from the potential alone.
2. A search over **all** Gaussian states, with no assumed shape and random starting points, finds the gap the formula predicts.
3. **Momentum directions are free**: widening the state along a momentum direction costs nothing, widening it along a position direction costs a lot.
4. The **sealed predictions**: values of $\Gamma$ for pendulum chains that were written to a file, timed and hashed *before* any optimization. The file is downloaded from the repository, its hash is checked, and $\Gamma$ is recomputed here.

**Runtime → Run all.**

[1] T. Namba, *The resolution gap in many degrees of freedom*, preprint (2026), [doi:10.5281/zenodo.23371932](https://doi.org/10.5281/zenodo.23371932) · code: [github.com/neginyan/resolution-gap-many-dof](https://github.com/neginyan/resolution-gap-many-dof) ([doi:10.5281/zenodo.23370396](https://doi.org/10.5281/zenodo.23370396))

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time, json, hashlib, itertools
from scipy.optimize import minimize
from scipy.linalg import sqrtm

eps = 0.3      # coupling of the two pendulums (try 0.1 or 1)

## 1. The formula, from the potential alone

For $H = |p|^2/2 + V(q)$ the symmetric part of the flow's Jacobian has the blocks $M(q) = (I - V''(q))/2$, and $\kappa = \max_q \lambda_{\max} M(q)$. At the maximum $q_0$, with $e_+$ the top eigenvector of $M(q_0)$, the strain along $e_+$ is $g(q) = e_+^{\mathsf T}M(q)e_+$, and
$$B_q = -\tfrac12\nabla^2 g(q_0),\qquad B = P^{\mathsf T}B_qP,\quad P = (e_+/\sqrt2,\, e_2,\dots,e_n).$$
The function below does exactly this, numerically, for any potential given by its Hessian $V''(q)$. Nothing in it is specific to pendulums.

In [ ]:
def Gamma_from_potential(Vpp, n, n_random=200, seed=0):
    # kappa and q0: maximise lambda_max(M(q)) from the corners {0, pi}^n and random points
    M = lambda q: (np.eye(n) - Vpp(q)) / 2
    lam = lambda q: -np.linalg.eigvalsh(M(q))[-1]
    rng = np.random.default_rng(seed)
    starts = [np.array(c, float) + 1e-3 for c in itertools.product([0, np.pi], repeat=n)] + list(rng.uniform(0, 2*np.pi, (n_random, n)))
    best = min((minimize(lam, q, method='Nelder-Mead', options={'xatol': 1e-12, 'fatol': 1e-15, 'maxiter': 20000}) for q in starts), key=lambda r: r.fun)
    q0 = best.x; kap = -best.fun
    w, U = np.linalg.eigh(M(q0)); ep = U[:, -1]
    g = lambda q: ep @ M(q) @ ep
    h = 1e-4; H = np.zeros((n, n))                       # Hessian of g by central differences
    for i in range(n):
        for j in range(n):
            ei = np.eye(n)[i] * h; ej = np.eye(n)[j] * h
            H[i, j] = (g(q0+ei+ej) - g(q0+ei-ej) - g(q0-ei+ej) + g(q0-ei-ej)) / (4*h*h)
    Bq = -H / 2
    P = np.column_stack([ep / np.sqrt(2)] + [U[:, k] for k in range(n-1)])
    B = P.T @ Bq @ P; B = (B + B.T) / 2
    C = np.diag([2*kap] + [kap]*(n-1))
    Ch = np.sqrt(C); G = 2*np.trace(np.real(sqrtm(Ch @ B @ Ch)))
    return dict(kappa=kap, q0=np.mod(q0 + 1e-9, 2*np.pi), Gamma=G, B=B)

def Vpp_coupled(e):          # V = -cos q1 - cos q2 - e cos(q1 - q2)
    def f(q):
        c12 = np.cos(q[0]-q[1])
        return np.array([[np.cos(q[0]) + e*c12, -e*c12], [-e*c12, np.cos(q[1]) + e*c12]])
    return f

res = Gamma_from_potential(Vpp_coupled(eps), 2)
k_exact = (1 + eps + np.sqrt(1 + eps**2)) / 2
print(f"two coupled pendulums, eps = {eps}")
print(f"  kappa = {res['kappa']:.10f}   (closed form (1 + eps + sqrt(1 + eps^2))/2 = {k_exact:.10f})")
print(f"  top of the strain at q0/pi = {np.round(res['q0']/np.pi, 6) % 2}")
print(f"  Gamma = {res['Gamma']:.7f}")
print(f"  weak-coupling series 1 + sqrt(eps) + 3eps/4 + eps^1.5/8 = {1 + np.sqrt(eps) + 0.75*eps + eps**1.5/8:.7f}  (good for small eps)")
Gamma = res['Gamma']

## 2. A search over all Gaussian states, with no assumed shape

For a Gaussian state $\mathcal N(m, S)$ the critical rate is exact [1, 2]:
$$\kappa - R = \frac{\mathrm{tr}\big(F(\kappa I - K)\big)}{\mathrm{tr}\,F},\qquad F = (S + \sigma^2 I)^{-1},\quad K = \begin{pmatrix}0 & \bar M\\ \bar M & 0\end{pmatrix},\quad \bar M = \mathbb E[M(q)],$$
and for trigonometric potentials $\bar M$ is a closed form ($\mathbb E\cos q_i = \cos m_i\, e^{-S_{ii}/2}$).

The search below knows nothing about the formula: the mean is placed at random anywhere on the torus, and the covariance $S = LL^{\mathsf T}$ is a random $4\times4$ factor in the plain coordinates $(q_1, q_2, p_1, p_2)$. Each start is followed downhill (L-BFGS). Many starts end at other local optima (for example a state with only one pendulum near its top); the question is whether the **lowest** gap found agrees with $\Gamma$.

At finite $\sigma$ the optimum lies slightly below $\Gamma$, by an amount proportional to $\sigma$ (the formula is the leading order). So the best state found at $\sigma = 10^{-2}$ is then followed down to $\sigma = 3\times10^{-3}$ and $10^{-3}$, and the straight line through the last two points is extended to $\sigma = 0$.

In [ ]:
def EVpp_gauss(m, S, e):     # E[V''(q)] for q ~ N(m, S_qq), closed form
    c1 = np.cos(m[0]) * np.exp(-S[0, 0]/2); c2 = np.cos(m[1]) * np.exp(-S[1, 1]/2)
    c12 = np.cos(m[0]-m[1]) * np.exp(-(S[0, 0] + S[1, 1] - 2*S[0, 1])/2)
    return np.array([[c1 + e*c12, -e*c12], [-e*c12, c2 + e*c12]])

def unpack(x):
    m = x[:2]; L = np.zeros((4, 4)); L[np.tril_indices(4)] = x[2:12]
    L[np.diag_indices(4)] = np.exp(x[[2, 4, 7, 11]])
    return m, L

def gap_over_sigma(x, s, e):
    m, L = unpack(x); S = L @ L.T; k = (1 + e + np.sqrt(1 + e*e)) / 2
    Mb = (np.eye(2) - EVpp_gauss(m, S, e)) / 2
    K = np.block([[np.zeros((2, 2)), Mb], [Mb, np.zeros((2, 2))]])
    _, Rr = np.linalg.qr(np.vstack([L.T, s*np.eye(4)]))      # S + s^2 I = Rr^T Rr, stable for thin states
    Ri = np.linalg.inv(Rr); F = Ri @ Ri.T
    return float(np.trace(F @ (k*np.eye(4) - K)) / np.trace(F)) / s

def bounds(s):
    lo = np.full(12, -30.); hi = np.full(12, 30.)
    lo[[2, 4, 7, 11]] = np.log(s) - 25; hi[[2, 4, 7, 11]] = 6; lo[:2] = -10; hi[:2] = 20
    return list(zip(lo, hi))

def polish(x, s, e):
    f = lambda y: gap_over_sigma(y, s, e)
    r = minimize(f, x, method='L-BFGS-B', bounds=bounds(s), options={'maxiter': 3000, 'maxfun': 60000})
    r = minimize(f, r.x, method='L-BFGS-B', bounds=bounds(s), options={'maxiter': 3000, 'maxfun': 60000, 'ftol': 1e-15, 'gtol': 1e-12})
    return r.fun, r.x

rng = np.random.default_rng(1); s = 1e-2; found = []; t0 = time.time()
for i in range(40):
    x0 = np.zeros(12); x0[:2] = rng.uniform(0, 2*np.pi, 2)
    x0[[2, 4, 7, 11]] = rng.uniform(np.log(s) - 2, 1, 4); x0[[3, 5, 6, 8, 9, 10]] = rng.normal(0, 0.3, 6)
    found.append(polish(x0, s, eps))
found.sort(key=lambda t: t[0])
print(f"40 random starts at sigma = 1e-2 ({time.time() - t0:.0f} s); lowest gaps/sigma found:", np.round([v for v, _ in found[:5]], 5))
print(f"  {sum(v < found[0][0] + 1e-3 for v, _ in found)} of 40 starts reached the lowest value")

x = found[0][1]; follow = [(s, found[0][0])]
for s_next in [3e-3, 1e-3]:
    v, x = polish(x, s_next, eps)
    r = minimize(lambda y: gap_over_sigma(y, s_next, eps), x, method='Powell', options={'xtol': 1e-12, 'ftol': 1e-15, 'maxfev': 40000})
    if r.fun < v: v, x = r.fun, r.x
    follow.append((s_next, v))
print("\nfollowed down in sigma:")
for s_, v in follow:
    print(f"  sigma = {s_:.0e}:  (kappa - R*)/sigma = {v:.6f}   difference from Gamma = {v - Gamma:+.5f} = {(v - Gamma)/s_:+.2f} sigma")
(s1, v1), (s2, v2) = follow[-2], follow[-1]
G_extrap = v2 - s2 * (v1 - v2) / (s1 - s2)            # straight line through the last two points, to sigma = 0
print(f"\nextrapolated to sigma -> 0: {G_extrap:.6f}     formula: Gamma = {Gamma:.6f}     difference {G_extrap - Gamma:+.1e}")
x_best = x

The lowest gap found by the blind search differs from $\Gamma$ by an amount proportional to $\sigma$ (the finite-resolution correction, about $-2.4\,\sigma$ for $\varepsilon = 0.3$), and extended to $\sigma = 0$ it lands on $\Gamma$.

What does the winning state look like? Its mean sits at the top of the strain, $q_0$, and it is a thin needle across the most stretching direction, long along the contracting one and along the second pendulum's position, and **very long along the momentum of the second mode**, as the paper predicts.

In [ ]:
m, L = unpack(x_best); S = L @ L.T
print("mean q / pi =", np.round(m / np.pi, 4) % 2, "   (tops of the strain: (1, 0) and, by the symmetry of the two pendulums, (0, 1))")
w, U = np.linalg.eigh(S)
print("standard deviations along the principal axes of S:", np.array2string(np.sqrt(np.maximum(w, 0)), precision=4))
print("  (sigma = 1e-3: one axis far thinner than sigma, two of order sqrt(sigma) = 0.03, one at the size limit of the search)")

## 3. Momentum directions are free

The strain felt by a Gaussian state depends only on its spread in **configuration space** (Proposition 1 of [1]). So the observer can let the state be long along a momentum direction at no cost in strain, while gaining focus elsewhere. Widening it along a configuration direction, by contrast, moves part of the state away from the top of the strain and costs a lot.

Below, the optimal state at $\sigma = 10^{-3}$ is widened by an extra standard deviation $\delta$ along the position, or along the momentum, of the **second mode** (the eigenvector $e_2$ of $M(q_0)$).

In [ ]:
s = 1e-3
m, L = unpack(x_best); S0 = L @ L.T
k = (1 + eps + np.sqrt(1 + eps**2)) / 2
def gap_S(S, s):
    Mb = (np.eye(2) - EVpp_gauss(m, S, eps)) / 2
    K = np.block([[np.zeros((2, 2)), Mb], [Mb, np.zeros((2, 2))]])
    F = np.linalg.inv(S + s*s*np.eye(4)); return float(np.trace(F @ (k*np.eye(4) - K)) / np.trace(F)) / s
e2 = np.linalg.eigh((np.eye(2) - Vpp_coupled(eps)(np.mod(m, 2*np.pi))) / 2)[1][:, 0]
u_q = np.concatenate([e2, [0, 0]]); u_p = np.concatenate([[0, 0], e2])
deltas = np.linspace(0, 0.3, 31)
gq = [gap_S(S0 + d*d*np.outer(u_q, u_q), s) for d in deltas]
gp = [gap_S(S0 + d*d*np.outer(u_p, u_p), s) for d in deltas]
print(f"gap/sigma of the optimal state: {gap_S(S0, s):.5f}")
print(f"widened by delta = 0.3 along the second mode's position: {gq[-1]:.3f}")
print(f"widened by delta = 0.3 along the second mode's momentum: {gp[-1]:.5f}")

## 4. The sealed predictions

The formula was found while studying two coupled pendulums. To test it on systems that played no part in its derivation, its values for open chains of 3 and 4 pendulums,
$$V = -\sum_i \cos q_i - \varepsilon\sum_i \cos(q_i - q_{i+1}),$$
and for a product coupling were written to a file **before** any of those systems was optimized; the time and the SHA-256 hash of the file were logged. All six predictions were later confirmed by full optimization (Table II of [1]).

Here the file is downloaded from the repository, its hash is checked against the one logged at the time, and the predictions for the chains are recomputed with the function of Section 1.

In [ ]:
URL = "https://raw.githubusercontent.com/neginyan/resolution-gap-many-dof/main/study/results/m2_prediction.json"
LOGGED = "d84ba44aab9c4e5b09a2079589db0f6f69770ff314f08a92f7562139cfed7664"   # study/logs/m2_prediction_hash.txt
optimum_found_later = {(3, 0.1): 1.8198116, (3, 0.3): 2.7601116, (4, 0.1): 1.8798244, (4, 0.3): 2.9974828}   # Table II of [1]
try:
    import urllib.request
    raw = urllib.request.urlopen(URL, timeout=20).read()
    print("downloaded; SHA-256 =", hashlib.sha256(raw).hexdigest())
    print("matches the hash logged on 2026-10-13T00:46:28Z:", hashlib.sha256(raw).hexdigest() == LOGGED)
    pred = json.loads(raw); print("written:", pred["written_utc"])
    sealed = {(d["n"], d["eps"]): d["prediction"] for d in pred["systems"] if d["kind"] == "chain"}
except Exception as err:
    print("could not download the file (", err, "); using the values printed in the paper")
    sealed = {(3, 0.1): 1.8198116, (3, 0.3): 2.7601116, (4, 0.1): 1.8798244, (4, 0.3): 2.9974827}

def Vpp_chain(n, e):
    def f(q):
        H = np.diag(np.cos(q)).astype(float)
        for i in range(n - 1):
            c = e*np.cos(q[i] - q[i+1]); H[i, i] += c; H[i+1, i+1] += c; H[i, i+1] -= c; H[i+1, i] -= c
        return H
    return f

print("\n chain           sealed prediction   recomputed here   optimum found later")
for (n, e) in [(3, 0.1), (3, 0.3), (4, 0.1), (4, 0.3)]:
    G = Gamma_from_potential(Vpp_chain(n, e), n, n_random=60)["Gamma"]
    print(f" n = {n}, eps = {e}     {sealed[(n, e)]:.7f}          {G:.7f}         {optimum_found_later[(n, e)]:.7f}")

## 5. One picture

Left: $\Gamma$ against the coupling, with the optimum found by full optimization in the paper (circles) and the blind search of this notebook (star). Right: the cost of widening the optimal state along a position direction and along a momentum direction.

In [ ]:
e_grid = np.linspace(0.02, 3, 60)
G_grid = [Gamma_from_potential(Vpp_coupled(e), 2, n_random=20)["Gamma"] for e in e_grid]
table = {0.1: 1.4025145, 0.3: 1.8474770, 1: 3.2537898, 2: 5.2354984, 3: 7.2201064}   # Table I of [1], optimum extrapolated to sigma -> 0

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].plot(e_grid, G_grid, color='#1f4e79', lw=2, label=r'formula $\Gamma = 2\,\mathrm{tr}[(B^{1/2}CB^{1/2})^{1/2}]$')
e_s = e_grid[e_grid <= 0.6]
ax[0].plot(e_s, 1 + np.sqrt(e_s) + 0.75*e_s + e_s**1.5/8, ':', color='gray', label=r'weak coupling $1+\sqrt{\varepsilon}+\frac{3}{4}\varepsilon+\frac{1}{8}\varepsilon^{3/2}$')
ax[0].plot(list(table), list(table.values()), 'o', color='#c55a11', ms=8, mfc='none', mew=2, label='full optimization (paper, Table I)')
ax[0].plot([eps], [G_extrap], '*', color='#2e7d32', ms=16, label=r'blind search here, extrapolated to $\sigma\to0$')
ax[0].set_xlabel(r'coupling $\varepsilon$'); ax[0].set_ylabel(r'gap / $\sigma$ of the best Gaussian state')
ax[0].set_title('The formula against the optimum'); ax[0].legend(fontsize=8.5); ax[0].set_ylim(0.8, 7.8)
ax[1].plot(deltas, gq, color='#c55a11', lw=2, label="widen along the 2nd mode's position")
ax[1].plot(deltas, gp, color='#1f4e79', lw=2, label="widen along the 2nd mode's momentum")
ax[1].set_xlabel(r'extra width $\delta$'); ax[1].set_ylabel(r'gap / $\sigma$  ($\sigma = 10^{-3}$)')
ax[1].set_title('Momentum directions are free'); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig('resolution_gap_many_dof.png', dpi=150); plt.show()

**What this shows.** The gap of the best Gaussian state is set by one matrix formula. It does not grow with the number of degrees of freedom as such: an observer focused on one unstable mode can spread freely along every momentum and along modes that do not affect the maximal strain, and pays only for configuration directions on which the maximal strain depends.

[2] T. Namba, *A resolution gap for the coarse-grained arrow of time in nonlinear Hamiltonian flows*, preprint v1.1 (2026), [doi:10.5281/zenodo.23320603](https://doi.org/10.5281/zenodo.23320603).